<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/chunking_strategies_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub langchain_text_splitters langchain_core langchain-experimental langchain_huggingface torch tiktoken datasets

In [ ]:
from huggingface_hub import login


hf_token = ""

# Colab Secrets (🔑 in the sidebar)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass


login(token=hf_token)

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.3"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

In [ ]:
sample_text = train_df.loc[train_df["case_text"].str.len().idxmax(), "case_text"]

print(sample_text)

Structured Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re

def perform_semantic_chunking(document, chunk_size=500, chunk_overlap=100):
    # Create the text splitter with semantic separators
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", r"(?<=[.!?])\s+", " ", ""],     # Look behind regular expression.
        is_separator_regex=True,                                  # Treat separators as regular expressions.
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    # Split the text into semantic chunks
    semantic_chunks = text_splitter.split_text(document)
    print(f"Document split into {len(semantic_chunks)} semantic chunks")

    # Determine section titles for enhanced metadata
    section_patterns = [
        r'^#+\s+(.+)$',      # Markdown headers
        r'^[A-Z\s]+:$'       # ALL CAPS section titles
    ]

    # Convert to Document objects with enhanced metadata
    documents = []
    current_section = "Introduction"

    for i, chunk in enumerate(semantic_chunks):
        # Try to identify section title from chunk
        chunk_lines = chunk.split('\n')
        for line in chunk_lines:
            for pattern in section_patterns:
                match = re.match(pattern, line, re.MULTILINE)
                if match:
                    current_section = match.group(0)
                    break

        # Calculate semantic density (ratio of non-stopwords to total words)
        words = re.findall(r'\b\w+\b', chunk.lower())
        stopwords = ['the', 'and', 'is', 'of', 'to', 'a', 'in', 'that', 'it', 'with', 'as', 'for']
        content_words = [w for w in words if w not in stopwords]
        semantic_density = len(content_words) / max(1, len(words))

        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "total_chunks": len(semantic_chunks),
                "chunk_size": len(chunk),
                "chunk_type": "semantic",
                "section": current_section,
                "semantic_density": round(semantic_density, 2)
            }
        )
        documents.append(doc)

    return documents



# Create the dummy document
document = sample_text

# Process with semantic chunking
chunked_docs = perform_semantic_chunking(document)

# Display results
print("\n----- CHUNKING RESULTS -----")
print(f"Total semantic chunks: {len(chunked_docs)}")

# Print an example chunk
print("\n----- EXAMPLE STRUCTURED CHUNK -----")
example_chunk = chunked_docs[0]
print("-" * 40)
print(example_chunk.page_content)
print("-" * 40)
print(f"Metadata: {example_chunk.metadata}")

# Optional: Calculate section distribution for analysis
section_counts = {}
for doc in chunked_docs:
    section = doc.metadata["section"]
    section_counts[section] = section_counts.get(section, 0) + 1

print("\n----- SECTION DISTRIBUTION -----")
for section, count in section_counts.items():
    print(f"{section}: {count} chunks")

In [ ]:
import torch
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # Choose embedding model you like on Huggingface
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)


splitter = SemanticChunker(embedding_model)

chunks = splitter.split_text(sample_text)

print("Semantic Chunks:", len(chunks))